# FX Pairs Trading — Model Training Pipeline V7
## Dataset-matched, leakage-safe target alignment, stronger validation, and robust model configuration

This version is rebuilt against the uploaded **82-column `dcc_garch_calander_gdelt_final_v4` dataset** (20 groups × 2,843 business dates).

**Forecasting objective:** predict the next 5 *future daily changes* in the VECM spread z-score.  
For time-series forecasting libraries, the response at timestamp `t` must be the change that *occurs at `t`*, i.e. `zscore[t] - zscore[t-1]`. Therefore V7 uses a correctly timestamped `target = groupby(zscore).diff()` rather than storing the next-row change on the current row.

Run cells top-to-bottom in Colab.


## V7 — Important fixes made from the actual uploaded CSV

1. **Target alignment fixed.** The old notebook used `groupby("zscore").diff().shift(-1)`. That is a valid next-row label for a tabular one-step model, but it is timestamp-misaligned for TFT/NeuralForecast multi-horizon forecasting. V7 uses `target = groupby("zscore").diff()` so the first future timestamp represents the immediate next daily z-score change.
2. **Actual 82-column schema used directly.** Removed assumptions from the prior ~616-column schema.
3. **No nonexistent `event_lag_2`.** The uploaded file contains only `event_lag_1`.
4. **Explicit future leakage blocked.** `macro_1_lead1`, `macro_2_lead1`, and `macro_3_lead1` are never used as model inputs.
5. **Strong historical features restored.** The pipeline now uses z-score/spread, returns, rolling return statistics, DCC-GARCH sigma/rho, macro, GDELT, regimes, and event history.
6. **Regimes one-hot encoded.** Avoids imposing false ordinal distances on `regime`, `regime_vol`, and `regime_macro`.
7. **Calendar features expanded.** Day-of-week, month, and day-of-year cyclical features are generated from `Date` and treated as known-future inputs.
8. **TFT validation improved.** Validation now evaluates rolling windows across the declared validation period instead of only the single last prediction window.
9. **NeuralForecast future exog fixed.** When calendar variables are enabled, `.predict()` now receives the held-out future calendar frame explicitly.
10. **PatchTST made API-robust.** Constructor arguments and exogenous inputs are passed only when the installed NeuralForecast version supports them.
11. **NBEATSx exogenous stack enabled** when exogenous inputs are supported.
12. **Zero-change baseline + RMSE + directional accuracy** are added so a noisy FX-change model is not judged by MAE alone.
13. **Longer encoder / stronger regularization.** Encoder length is 120 business days, validation window 120 days, and training budgets are increased with early stopping.

> Important: higher training capacity can improve fit, but no notebook can guarantee higher held-out performance. Trust the held-out leaderboard and the zero-change baseline, not training loss.


## Dataset schema used by V7

The uploaded CSV contains 82 source columns, including:
- identifiers: `Date`, `group`, `group_id`, `pair_1..3`, `time_idx`
- spread/return state: `zscore`, `spread`, returns, rolling means/stds, `zscore_lag1`, `zscore_diff_1`, `zscore_diff_5`
- DCC-GARCH state: `sigma_1..3`, `rho_12/13/23`, rho changes, `sigma_group`
- macro state: `macro_1..3`, surprises, rates, inflation, labor, `macro_pressure`, `regime_macro`
- explicit future macro fields: `macro_1_lead1`, `macro_2_lead1`, `macro_3_lead1` (**excluded**)
- GDELT/news state: tone, Goldstein, shock, mentions, rolling news summaries/interactions, `event_flag`, `event_lag_1`
- regimes: `regime`, `regime_vol`


## Evaluation principle

The final 5 business days of each group are held out once as the test set.  
The preceding 120 business days per group are validation data. Hyperparameters/early stopping must use validation only; the test set is for final scoring.

Because the response is a noisy, approximately zero-centered daily change, V7 always compares deep models against a **zero-change forecast**. A model that cannot beat this baseline on held-out MAE is not yet useful, even if its training loss looks good.


## Model Training & Evaluation

Models:
- Temporal Fusion Transformer (TFT)
- NBEATSx
- NHITS
- PatchTST
- Spacetimeformer (optional control; runs only if a compatible package is already installed)

Primary metric: **MAE**  
Additional metrics: **RMSE** and **directional accuracy** (sign hit-rate).


In [ ]:
!pip install -q lightning neuralforecast pytorch_forecasting pyarrow


### Step 0b — Mount Drive, global config, folder structure

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell 0 — Mount + global config + folder structure
# ══════════════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, glob, random
import numpy as np
import torch

drive_base = "/content/drive/MyDrive/fx_models_v7/"
DATA_DIR   = f"{drive_base}data/"
for sub in ["tft", "nbeats", "nhits", "patchtst", "spacetimeformer", "ensemble", "leaderboard"]:
    os.makedirs(f"{drive_base}{sub}/", exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

# Prefer the exact uploaded filename, but fall back to common variants.
_csv_candidates = [
    "/content/drive/MyDrive/dcc_garch_calander_gdelt_final_v4 (1).csv",
    "/content/drive/MyDrive/dcc_garch_calander_gdelt_final_v4.csv",
    "/content/drive/MyDrive/dcc_garch_calander_gdelt_final.csv",
]
CSV_PATH = next((p for p in _csv_candidates if os.path.exists(p)), None)
if CSV_PATH is None:
    raise FileNotFoundError(
        "Dataset not found in MyDrive. Upload/copy the V4 CSV to MyDrive, or edit "
        "_csv_candidates in this cell with its exact path."
    )

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

try:
    import lightning.pytorch as pl
    pl.seed_everything(SEED, workers=True)
except ImportError:
    pass

print(f"✅ CSV: {CSV_PATH}")
print(f"✅ Output root: {drive_base}")
print(f"✅ Global random seed: {SEED}")


### Step 1 — Load data, build target, chronological per-group split, save splits

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell A — Load data, ALIGN target correctly, chronological split, save
# ══════════════════════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
import json
import pickle

df_raw = pd.read_csv(CSV_PATH)
df_raw["Date"] = pd.to_datetime(df_raw["Date"])
df_raw = df_raw.sort_values(["group_id", "Date"]).reset_index(drop=True)

required = {"Date", "group_id", "zscore"}
missing_required = required.difference(df_raw.columns)
assert not missing_required, f"Missing required columns: {sorted(missing_required)}"

assert not df_raw.duplicated(["group_id", "Date"]).any(),     "Duplicate (group_id, Date) rows found; fix the dataset before training."

# ── TARGET ALIGNMENT FIX ──────────────────────────────────────────────────
# y_t = zscore_t - zscore_(t-1)
# Forecasting libraries attach y to its own timestamp. Therefore predicting
# the next timestamp gives the immediate next daily z-score change.
df_raw["target"] = df_raw.groupby("group_id", sort=False)["zscore"].diff()

# Verify source zscore_diff_1 if present. Its first row/group is initialized
# to 0 in this CSV; all non-initial rows should match our recomputation.
if "zscore_diff_1" in df_raw.columns:
    _check = df_raw["target"].notna()
    _max_err = (df_raw.loc[_check, "target"] - df_raw.loc[_check, "zscore_diff_1"]).abs().max()
    assert _max_err < 1e-10, (
        f"zscore_diff_1 does not match recomputed groupwise z-score change; max error={_max_err}"
    )
    print("✅ target verified against source zscore_diff_1 (non-initial rows)")

# Drop the first row of each series where a genuine change is undefined.
df_raw = df_raw.dropna(subset=["target"]).reset_index(drop=True)

# Stronger context / validation than V6.
max_encoder_length = 120
horizon = 5
val_window = 120

# Require a balanced panel because downstream TFT/Spacetimeformer logic relies
# on aligned group lengths.
_group_sizes = df_raw.groupby("group_id").size()
assert _group_sizes.nunique() == 1, (
    "Groups have unequal lengths. Use a per-group split/window implementation "
    "before training this notebook."
)

# Chronological split inside every group.
_pos = df_raw.groupby("group_id").cumcount()
_n   = df_raw.groupby("group_id")["group_id"].transform("size")
df_raw["split"] = "train"
df_raw.loc[_pos >= (_n - horizon - val_window), "split"] = "val"
df_raw.loc[_pos >= (_n - horizon), "split"] = "test"

df = df_raw.copy()

# Sanity checks
assert (df.groupby("group_id").size() >= max_encoder_length + val_window + horizon).all()
for split_name in ["train", "val", "test"]:
    sub = df[df["split"] == split_name]
    counts = sub.groupby("group_id").size()
    print(
        f"{split_name:5s}: rows={len(sub):6d} | "
        f"{sub['Date'].min().date()} -> {sub['Date'].max().date()} | "
        f"rows/group={counts.min()}..{counts.max()}"
    )

# Held-out zero-change baseline (do not tune to this; report only).
_test_target = df.loc[df["split"] == "test", "target"]
print(f"Zero-change held-out baseline MAE: {_test_target.abs().mean():.6f}")

# SAVE raw/aligned split data.
df.to_parquet(f"{DATA_DIR}full_dataset.parquet", index=False)
for split_name in ["train", "val", "test"]:
    df[df["split"] == split_name].to_parquet(f"{DATA_DIR}{split_name}.parquet", index=False)

with open(f"{DATA_DIR}split_config.json", "w") as f:
    json.dump({
        "target_definition": "target_t = zscore_t - zscore_(t-1)",
        "max_encoder_length": max_encoder_length,
        "horizon": horizon,
        "val_window": val_window,
        "csv_source": CSV_PATH,
        "n_groups": int(df["group_id"].nunique()),
        "rows_per_group": int(df.groupby("group_id").size().iloc[0]),
        "date_min": str(df["Date"].min().date()),
        "date_max": str(df["Date"].max().date()),
    }, f, indent=2)

print(f"✅ Saved aligned full/train/val/test data to {DATA_DIR}")


### Step 1b — (Optional) Reload from saved splits

Only needed if you're resuming a session instead of running Step 1 fresh in this runtime.

In [ ]:
df = pd.read_parquet(f"{DATA_DIR}full_dataset.parquet")
train_df = pd.read_parquet(f"{DATA_DIR}train.parquet")
val_df   = pd.read_parquet(f"{DATA_DIR}val.parquet")
test_df  = pd.read_parquet(f"{DATA_DIR}test.parquet")
with open(f"{DATA_DIR}split_config.json") as f:
    cfg = json.load(f)
max_encoder_length, horizon, val_window = cfg["max_encoder_length"], cfg["horizon"], cfg["val_window"]


### Step 2 — Feature column lists shared across all models

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell B — Feature lists matched EXACTLY to the uploaded 82-column CSV
# ══════════════════════════════════════════════════════════════════════════

# ── Known-future calendar features ────────────────────────────────────────
# Generate them from Date so future test dates can be supplied explicitly.
df["dow_num"] = df["Date"].dt.dayofweek.astype(float)
df["dow_sin"] = np.sin(2 * np.pi * df["dow_num"] / 5.0)
df["dow_cos"] = np.cos(2 * np.pi * df["dow_num"] / 5.0)
df["month_num"] = df["Date"].dt.month.astype(float)
df["month_sin"] = np.sin(2 * np.pi * (df["month_num"] - 1) / 12.0)
df["month_cos"] = np.cos(2 * np.pi * (df["month_num"] - 1) / 12.0)
df["doy_num"] = df["Date"].dt.dayofyear.astype(float)
df["doy_sin"] = np.sin(2 * np.pi * (df["doy_num"] - 1) / 365.25)
df["doy_cos"] = np.cos(2 * np.pi * (df["doy_num"] - 1) / 365.25)

calendar_cols = [
    "dow_sin", "dow_cos", "month_sin", "month_cos", "doy_sin", "doy_cos"
]

# ── Historical market / spread features ───────────────────────────────────
# zscore_diff_1 is NOT listed because it is the source-equivalent of target.
core_history_cols = [
    "zscore", "spread",
    "ret_1", "ret_2", "ret_3",
    "ret_1_lag1", "ret_2_lag1", "ret_3_lag1",
    "ret_1_mean_5", "ret_2_mean_5", "ret_3_mean_5",
    "ret_1_std_20", "ret_2_std_20", "ret_3_std_20",
    "sigma_1", "sigma_2", "sigma_3",
    "rho_12", "rho_13", "rho_23",
    "rho_12_change", "rho_13_change", "rho_23_change",
    "sigma_group",
    "zscore_lag1", "zscore_diff_5",
]
core_history_cols = [c for c in core_history_cols if c in df.columns]

# ── DCC-GARCH aliases retained for the optional Spacetimeformer/manifest ──
dcc_garch_leg_cols = [
    c for c in [
        "sigma_1", "sigma_2", "sigma_3",
        "rho_12", "rho_13", "rho_23",
        "rho_12_change", "rho_13_change", "rho_23_change",
        "sigma_group",
    ] if c in df.columns
]

# ── Macro features ─────────────────────────────────────────────────────────
# Explicit *_lead1 columns are future-shifted and forbidden.
LEAK_COLS = ["macro_1_lead1", "macro_2_lead1", "macro_3_lead1"]

macro_leg_cols = [
    c for c in df.columns
    if (
        c.startswith((
            "macro_1", "macro_2", "macro_3",
            "macro_surprise_1", "macro_surprise_2", "macro_surprise_3"
        ))
        and c not in LEAK_COLS
    )
]
global_macro_cols = [c for c in ["macro_pressure"] if c in df.columns]

# ── GDELT / event features ─────────────────────────────────────────────────
gdelt_cols = [
    c for c in df.columns
    if (
        c.startswith(("avg_tone_diff", "avg_goldstein_diff", "shock_rel", "mentions_rel"))
        or c == "tone_shock_interaction"
    )
]
extra_event_cols = [c for c in ["event_flag", "event_lag_1"] if c in df.columns]

# ── Regime features: one-hot, historical-only ─────────────────────────────
regime_cols = []
for base in ["regime", "regime_vol", "regime_macro"]:
    if base in df.columns:
        for value in sorted(df[base].dropna().unique().tolist()):
            value_label = str(int(value)) if float(value).is_integer() else str(value)
            out = f"{base}_{value_label}_oh"
            df[out] = (df[base] == value).astype(float)
            regime_cols.append(out)

# Build a single, deduplicated historical feature list.
hist_exog = list(dict.fromkeys(
    core_history_cols
    + macro_leg_cols
    + global_macro_cols
    + gdelt_cols
    + extra_event_cols
    + regime_cols
))

# Safety guards
for c in LEAK_COLS:
    assert c not in hist_exog, f"Leak column accidentally included: {c}"
assert "zscore_diff_1" not in hist_exog,     "zscore_diff_1 duplicates the response and must not be an exogenous feature."
assert "target" not in hist_exog
_missing = [c for c in hist_exog + calendar_cols if c not in df.columns]
assert not _missing, f"Selected feature columns missing from df: {_missing}"
_non_numeric = [c for c in hist_exog + calendar_cols if not pd.api.types.is_numeric_dtype(df[c])]
assert not _non_numeric, f"Non-numeric model features found: {_non_numeric}"
assert not df[hist_exog + calendar_cols + ["target"]].isna().any().any(),     "NaNs found in model inputs after feature construction."

# Aliases used by older optional cells / manifest.
unknown_cols = ["target"] + hist_exog
known_cols = calendar_cols
static_cols = ["group_id"]

print(f"✅ target: target_t = zscore_t - zscore_(t-1)")
print(f"✅ historical exogenous features: {len(hist_exog)}")
print(f"   market/spread: {len(core_history_cols)}")
print(f"   macro:         {len(macro_leg_cols) + len(global_macro_cols)}")
print(f"   GDELT/event:   {len(gdelt_cols) + len(extra_event_cols)}")
print(f"   regime OHE:    {len(regime_cols)}")
print(f"✅ known-future calendar features: {calendar_cols}")
print(f"🚫 explicit leak columns excluded: {LEAK_COLS}")


In [ ]:
# ── V7 feature audit ──────────────────────────────────────────────────────
# Run immediately after Cell B.
print("Dataset after derived features:", df.shape)
print("Groups:", df["group_id"].nunique())
print("Rows/group:", df.groupby("group_id").size().min(), "..", df.groupby("group_id").size().max())
print("Duplicate group-date rows:", int(df.duplicated(["group_id", "Date"]).sum()))

audit = pd.DataFrame({
    "dtype": df[hist_exog + calendar_cols + ["target"]].dtypes.astype(str),
    "nan_rate": df[hist_exog + calendar_cols + ["target"]].isna().mean(),
    "n_unique": df[hist_exog + calendar_cols + ["target"]].nunique(),
})
display(audit)

# Training-only lag-1 linear signal diagnostic. This is NOT feature selection;
# it is just a sanity check showing which historical fields have the strongest
# simple one-step relationship with the response.
_train = df[df["split"] == "train"].sort_values(["group_id", "Date"])
_lagged = _train.groupby("group_id")[hist_exog].shift(1)
_lag_corr = _lagged.corrwith(_train["target"]).abs().sort_values(ascending=False)
print("\nTop 15 absolute lag-1 correlations on TRAIN only:")
print(_lag_corr.head(15).to_string())


Clean file

In [ ]:
# ── Full-folder reset before a clean rerun ──────────────────────────────────
# Clears every model-output subfolder under drive_base so a rerun starts from
# nothing instead of accumulating old checkpoints/predictions/logs alongside
# new ones. Run this right after Cell 0 (Step 0b, which defines drive_base
# and recreates these folders), before Step 1.

import shutil, os

# Model-output folders only -- NOT DATA_DIR. DATA_DIR holds train/val/test
# parquet splits + split_config.json, which are expensive to regenerate and
# which Step 1b is specifically built to reload -- wiping those on every
# rerun would silently force a fresh chronological split each time, which is
# a much bigger change than "clean model artifacts."
CLEAR_SUBFOLDERS = ["tft", "nbeats", "nhits", "patchtst", "spacetimeformer", "ensemble", "leaderboard"]

# Set this True only if you also want DATA_DIR wiped -- i.e. you intend to
# rebuild the splits from scratch this run, not just retrain models.
ALSO_CLEAR_DATA_DIR = False

for sub in CLEAR_SUBFOLDERS:
    path = f"{drive_base}{sub}/"
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)
    print(f"Cleared {path}")

if ALSO_CLEAR_DATA_DIR and os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
    os.makedirs(DATA_DIR, exist_ok=True)
    print(f"Cleared {DATA_DIR} (ALSO_CLEAR_DATA_DIR=True)")
elif not ALSO_CLEAR_DATA_DIR:
    print(f"Left {DATA_DIR} untouched (ALSO_CLEAR_DATA_DIR=False) -- "
          f"train/val/test splits will be reused if present.")

# manifest.json / hyperparameters.json live directly under drive_base, not
# inside a subfolder -- these get overwritten (not accumulated) by Step 12
# regardless, so no separate clear needed for them.

### Step 3 — Train the Temporal Fusion Transformer (TFT)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell C — TFT: rolling validation over the declared validation window
# ══════════════════════════════════════════════════════════════════════════
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss, MAE as PF_MAE, RMSE as PF_RMSE
import torch.nn as nn
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.loggers import CSVLogger

tft_df = df.copy()
tft_df["group_id"] = tft_df["group_id"].astype(str)
tft_df["time_idx"] = tft_df.groupby("group_id").cumcount()

train_val_df = tft_df[tft_df["split"].isin(["train", "val"])].copy()

per_group_max = train_val_df.groupby("group_id")["time_idx"].max()
assert per_group_max.nunique() == 1, (
    "Groups have different train+val lengths; global TFT cutoff is unsafe."
)
training_cutoff = int(train_val_df["time_idx"].max() - val_window)

training_dataset = TimeSeriesDataSet(
    train_val_df[train_val_df["time_idx"] <= training_cutoff],
    time_idx="time_idx",
    target="target",
    group_ids=["group_id"],
    static_categoricals=["group_id"],
    max_encoder_length=max_encoder_length,
    max_prediction_length=horizon,
    time_varying_known_reals=calendar_cols,
    time_varying_unknown_reals=["target"] + hist_exog,
    target_normalizer=GroupNormalizer(groups=["group_id"]),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=False,
)

# IMPORTANT: predict=False + min_prediction_idx makes validation span rolling
# windows across the validation period, instead of validating only on one
# final 5-day window.
validation_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    train_val_df,
    min_prediction_idx=training_cutoff + 1,
    stop_randomization=True,
    predict=False,
)

with open(f"{drive_base}tft/dataset_params.pkl", "wb") as f:
    pickle.dump(training_dataset.get_parameters(), f)

train_loader = training_dataset.to_dataloader(
    train=True, batch_size=128, num_workers=2
)
val_loader = validation_dataset.to_dataloader(
    train=False, batch_size=128, num_workers=2
)

checkpoint_cb = ModelCheckpoint(
    dirpath=f"{drive_base}tft/checkpoints/",
    filename="best-{epoch}-{val_loss:.5f}",
    monitor="val_loss", mode="min", save_top_k=1,
)
early_stop_cb = EarlyStopping(
    monitor="val_loss", patience=8, min_delta=1e-5, mode="min"
)

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=5e-4,
    hidden_size=64,
    attention_head_size=4,
    dropout=0.15,
    hidden_continuous_size=32,
    loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]),
    logging_metrics=nn.ModuleList([PF_MAE(), PF_RMSE()]),
    log_interval=20,
    reduce_on_plateau_patience=3,
)

trainer = pl.Trainer(
    max_epochs=80,
    callbacks=[checkpoint_cb, early_stop_cb, LearningRateMonitor(logging_interval="epoch")],
    logger=CSVLogger(save_dir=f"{drive_base}tft/", name="logs"),
    gradient_clip_val=0.1,
    accelerator="auto",
    devices="auto",
)

trainer.fit(tft, train_dataloaders=train_loader, val_dataloaders=val_loader)

best_tft_path = checkpoint_cb.best_model_path
assert best_tft_path, "No TFT checkpoint was saved."
best_tft = TemporalFusionTransformer.load_from_checkpoint(best_tft_path)
print(f"✅ Best TFT checkpoint: {best_tft_path}")

tft_hparams = {
    "model": "TFT",
    "target": "target_t = zscore_t - zscore_(t-1)",
    "learning_rate": 5e-4,
    "hidden_size": 64,
    "attention_head_size": 4,
    "dropout": 0.15,
    "hidden_continuous_size": 32,
    "loss": "QuantileLoss([0.1,0.5,0.9])",
    "max_epochs": 80,
    "gradient_clip_val": 0.1,
    "early_stop_patience": 8,
    "max_encoder_length": max_encoder_length,
    "max_prediction_length": horizon,
    "val_window": val_window,
    "static_categoricals": ["group_id"],
    "time_varying_known_reals": calendar_cols,
    "time_varying_unknown_reals": ["target"] + hist_exog,
    "seed": SEED,
}
with open(f"{drive_base}tft/hyperparameters.json", "w") as f:
    json.dump(tft_hparams, f, indent=2)


### Step 4 — Evaluate TFT on the true held-out test window

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell D — TFT true held-out test evaluation
# ══════════════════════════════════════════════════════════════════════════
test_dataset = TimeSeriesDataSet.from_dataset(
    training_dataset,
    tft_df,
    predict=True,
    stop_randomization=True,
)
test_loader = test_dataset.to_dataloader(
    train=False, batch_size=128, num_workers=2
)

# Because the held-out test slice is exactly `horizon` rows/group and is at
# the end of each series, predict=True targets exactly that final test window.
test_metrics = trainer.test(best_tft, dataloaders=test_loader, verbose=True)

tft_preds = best_tft.predict(test_loader, mode="prediction", return_x=True)
with open(f"{drive_base}tft/test_predictions.pkl", "wb") as f:
    pickle.dump(tft_preds, f)
with open(f"{drive_base}tft/test_metrics.pkl", "wb") as f:
    pickle.dump(test_metrics, f)

print(f"✅ TFT held-out metrics + predictions saved to {drive_base}tft/")


### Step 5 — Shared helper: exact quantile-column lookup

Used by every NeuralForecast-family model below (NBEATSx, NHITS, PatchTST) so quantile columns are
matched exactly instead of via fragile substring checks like `"50" in c`.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Shared helper — robust NeuralForecast quantile-column lookup
# ──────────────────────────────────────────────────────────────────────────
import re

def pick_quantile_cols(preds_df: pd.DataFrame, model_name: str, quantiles=(0.1, 0.5, 0.9)):
    """
    Return {quantile: column_name} for NeuralForecast predictions.

    NeuralForecast/MQLoss naming has varied across versions. Common forms:
      model-median
      model-lo-80 / model-hi-80    for q=.10/.90
      model-10 / model-90
      model-q0.1 / model-q0.9

    This function tries exact semantic patterns first and refuses ambiguous
    matches instead of silently choosing the wrong interval side.
    """
    candidates = [c for c in preds_df.columns if str(c).startswith(model_name)]
    if not candidates:
        raise ValueError(
            f"No prediction columns start with {model_name!r}. "
            f"Available: {list(preds_df.columns)}"
        )

    out = {}
    for q in quantiles:
        q = float(q)

        if np.isclose(q, 0.5):
            matches = [
                c for c in candidates
                if str(c).lower().endswith("median")
                or re.search(r"(?:q|quantile)[-_]?0?\.5(?:0+)?$", str(c).lower())
            ]
        else:
            level = int(round(abs(q - 0.5) * 200))  # q=.1/.9 -> level 80
            side = "lo" if q < 0.5 else "hi"
            pct = int(round(q * 100))

            semantic = [
                c for c in candidates
                if re.search(rf"-{side}-{level}(?:\.0+)?$", str(c).lower())
            ]
            direct_pct = [
                c for c in candidates
                if re.search(rf"-(?:q[-_]?)?{pct}(?:\.0+)?$", str(c).lower())
            ]
            direct_q = [
                c for c in candidates
                if re.search(
                    rf"(?:q|quantile)[-_]?{re.escape(str(q))}(?:0+)?$",
                    str(c).lower()
                )
            ]
            matches = semantic or direct_q or direct_pct

        matches = list(dict.fromkeys(matches))
        if len(matches) != 1:
            raise ValueError(
                f"Could not uniquely identify q={q} for {model_name}. "
                f"Matches={matches}; available={candidates}"
            )
        out[q] = matches[0]

    return out


### Step 6 — Train NBEATSx, NHITS, PatchTST (NeuralForecast)

`val_size` is fixed to `val_window` (not `val_window + horizon`) so the internal validation window
matches the declared val split instead of silently eating `horizon` extra days out of train.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell I — NeuralForecast: dataset-matched exog + robust API handling
# ──────────────────────────────────────────────────────────────────────────
import os, json, inspect
import torch
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATSx, NHITS, PatchTST
from neuralforecast.losses.pytorch import MQLoss, MAE

nf_df = df[[
    "group_id", "Date", "target", "split"
] + hist_exog + calendar_cols].copy()

nf_df = nf_df.rename(columns={"group_id": "unique_id", "Date": "ds", "target": "y"})
nf_df["ds"] = pd.to_datetime(nf_df["ds"])

# Train + validation only. True test rows never enter fit().
nf_train = nf_df[nf_df["split"].isin(["train", "val"])].drop(columns=["split"])

# Calendar values for exactly the held-out future timestamps.
nf_future = nf_df[nf_df["split"] == "test"][["unique_id", "ds"] + calendar_cols].copy()
assert nf_future.groupby("unique_id").size().eq(horizon).all(),     "Each series must have exactly `horizon` future rows."
assert not nf_future[calendar_cols].isna().any().any()

def _ctor_params(cls):
    return inspect.signature(cls.__init__).parameters

def _filter_kwargs(cls, kwargs):
    # Keep only explicitly declared model parameters. Do NOT treat **kwargs
    # as permission to pass arbitrary model arguments: in several
    # NeuralForecast versions **trainer_kwargs is forwarded to Lightning,
    # where an unsupported model argument would fail later and opaquely.
    params = _ctor_params(cls)
    return {
        k: v for k, v in kwargs.items()
        if k in params and params[k].kind != inspect.Parameter.VAR_KEYWORD
    }

def _supports(cls, name):
    params = _ctor_params(cls)
    return name in params and params[name].kind != inspect.Parameter.VAR_KEYWORD

shared = dict(
    h=horizon,
    input_size=max_encoder_length,
    loss=MQLoss(quantiles=[0.1, 0.5, 0.9]),
    valid_loss=MAE(),
    max_steps=800,
    val_check_steps=50,
    early_stop_patience_steps=8,
    scaler_type="robust",
    learning_rate=5e-4,
    random_seed=SEED,
)

def _with_exog(cls, base):
    out = dict(base)
    if _supports(cls, "hist_exog_list"):
        out["hist_exog_list"] = hist_exog
    if _supports(cls, "futr_exog_list"):
        out["futr_exog_list"] = calendar_cols
    return out

# ── NBEATSx ────────────────────────────────────────────────────────────────
nbeats_kwargs = _with_exog(NBEATSx, shared)
if _supports(NBEATSx, "stack_types"):
    # Exogenous stack is required for NBEATSx to explicitly model exogenous
    # covariates in versions that expose this basis.
    if _supports(NBEATSx, "hist_exog_list") or _supports(NBEATSx, "futr_exog_list"):
        nbeats_kwargs["stack_types"] = ["identity", "trend", "seasonality", "exogenous"]
        nbeats_kwargs["n_blocks"] = [2, 2, 2, 2]
        nbeats_kwargs["mlp_units"] = [[512, 512]] * 4
    else:
        nbeats_kwargs["stack_types"] = ["identity", "trend", "seasonality"]
        nbeats_kwargs["n_blocks"] = [2, 2, 2]
        nbeats_kwargs["mlp_units"] = [[512, 512]] * 3
if _supports(NBEATSx, "dropout_prob_theta"):
    nbeats_kwargs["dropout_prob_theta"] = 0.15
nbeats_model = NBEATSx(**_filter_kwargs(NBEATSx, nbeats_kwargs))

# ── NHITS ──────────────────────────────────────────────────────────────────
nhits_kwargs = _with_exog(NHITS, shared)
nhits_kwargs.update(
    n_freq_downsample=[8, 4, 1],
    n_blocks=[2, 2, 2],
    mlp_units=[[512, 512]] * 3,
)
if _supports(NHITS, "dropout_prob_theta"):
    nhits_kwargs["dropout_prob_theta"] = 0.15
nhits_model = NHITS(**_filter_kwargs(NHITS, nhits_kwargs))

# ── PatchTST ───────────────────────────────────────────────────────────────
# NeuralForecast changed some PatchTST parameter names across versions.
# Detect the installed signature instead of hardcoding incompatible names.
patch_kwargs = _with_exog(PatchTST, shared)
patch_kwargs.update(patch_len=16, stride=8, n_heads=8)
_patch_params = _ctor_params(PatchTST)
if "hidden_size" in _patch_params:
    patch_kwargs["hidden_size"] = 128
elif "d_model" in _patch_params:
    patch_kwargs["d_model"] = 128
if "encoder_layers" in _patch_params:
    patch_kwargs["encoder_layers"] = 3
elif "e_layers" in _patch_params:
    patch_kwargs["e_layers"] = 3
for name, value in [("dropout", 0.15), ("fc_dropout", 0.15), ("head_dropout", 0.10)]:
    if name in _patch_params:
        patch_kwargs[name] = value
patchtst_model = PatchTST(**_filter_kwargs(PatchTST, patch_kwargs))

nf_feature_mode = {
    "NBEATSx": {
        "hist_exog": bool(_supports(NBEATSx, "hist_exog_list")),
        "futr_exog": bool(_supports(NBEATSx, "futr_exog_list")),
    },
    "NHITS": {
        "hist_exog": bool(_supports(NHITS, "hist_exog_list")),
        "futr_exog": bool(_supports(NHITS, "futr_exog_list")),
    },
    "PatchTST": {
        "hist_exog": bool(_supports(PatchTST, "hist_exog_list")),
        "futr_exog": bool(_supports(PatchTST, "futr_exog_list")),
    },
}
print("NeuralForecast exogenous support detected:", nf_feature_mode)

nf_nbeats   = NeuralForecast(models=[nbeats_model],   freq="B")
nf_nhits    = NeuralForecast(models=[nhits_model],    freq="B")
nf_patchtst = NeuralForecast(models=[patchtst_model], freq="B")

def _fit_predict(nf_obj, model_name):
    nf_obj.fit(df=nf_train, val_size=val_window)
    if nf_feature_mode[model_name]["futr_exog"]:
        return nf_obj.predict(futr_df=nf_future)
    return nf_obj.predict()

nbeats_preds = _fit_predict(nf_nbeats, "NBEATSx")
nhits_preds = _fit_predict(nf_nhits, "NHITS")
patchtst_preds = _fit_predict(nf_patchtst, "PatchTST")

for name, nf_obj in [("nbeats", nf_nbeats), ("nhits", nf_nhits), ("patchtst", nf_patchtst)]:
    os.makedirs(f"{drive_base}{name}/", exist_ok=True)
    nf_obj.save(f"{drive_base}{name}/", overwrite=True)

nbeats_preds.to_csv(f"{drive_base}nbeats/test_predictions.csv", index=False)
nhits_preds.to_csv(f"{drive_base}nhits/test_predictions.csv", index=False)
patchtst_preds.to_csv(f"{drive_base}patchtst/test_predictions.csv", index=False)

nf_hparams = {
    "shared": {
        "target": "target_t = zscore_t - zscore_(t-1)",
        "h": horizon,
        "input_size": max_encoder_length,
        "calendar_cols": calendar_cols,
        "hist_exog": hist_exog,
        "loss": "MQLoss([0.1,0.5,0.9])",
        "valid_loss": "MAE",
        "max_steps": 800,
        "learning_rate": 5e-4,
        "val_check_steps": 50,
        "early_stop_patience_steps": 8,
        "scaler_type": "robust",
        "val_size": val_window,
        "seed": SEED,
        "feature_support_detected": nf_feature_mode,
    },
    "NBEATSx": {k: str(v) for k, v in _filter_kwargs(NBEATSx, nbeats_kwargs).items()
                 if k not in {"loss", "valid_loss", "callbacks", "trainer_kwargs"}},
    "NHITS": {k: str(v) for k, v in _filter_kwargs(NHITS, nhits_kwargs).items()
              if k not in {"loss", "valid_loss", "callbacks", "trainer_kwargs"}},
    "PatchTST": {k: str(v) for k, v in _filter_kwargs(PatchTST, patch_kwargs).items()
                 if k not in {"loss", "valid_loss", "callbacks", "trainer_kwargs"}},
}
for name in ["nbeats", "nhits", "patchtst"]:
    with open(f"{drive_base}{name}/hyperparameters.json", "w") as f:
        json.dump(nf_hparams, f, indent=2)

print("✅ NBEATSx / NHITS / PatchTST trained without test leakage.")


### Step 7 — Train Spacetimeformer (train+val only)

Built from train+val only, exactly like `nf_train`, so its "val" slice can never contain true test
rows. Checkpoints on best `val_loss` and loads that checkpoint before saving / evaluating.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell J — Spacetimeformer, train+val ONLY   (V4)
# ──────────────────────────────────────────────────────────────────────────
# WHY THIS ISN'T A CLEAN "ADD gdelt_cols + calendar_cols" LIKE TFT/NF:
#
# TFT and the NeuralForecast models already accept named exogenous-feature
# lists -- adding calendar/GDELT there is wiring, not architecture change.
# Spacetimeformer here is different. The class actually used --
# `stf.model.Spacetimeformer(d_x=n_pairs, d_y=n_pairs, max_seq_len=...,
# out_len=...)` with a single-tensor `forward(X)` call -- does NOT match the
# published `spacetimeformer` (QData) library's real API, which is
# `Spacetimeformer_Forecaster` taking separate `x_context, y_context,
# x_target, y_target` tensors. That means either a custom/simplified wrapper
# is installed in this environment, or there's a version mismatch -- and I
# could not confirm which from inside this notebook (it's installed outside
# Cell 2's `!pip install` line, so its actual signature is opaque here).
#
# CONCRETELY: does the class in *your* environment support d_x != d_y (more
# input channels -- target + exogenous -- than output channels -- target
# only)? I don't know, and guessing wrong here either crashes (safe) or
# silently trains on a broadcast/reshape that doesn't mean what you think it
# means (not safe, and much worse than a crash).
#
# BEFORE flipping STF_USE_EXOG to True below, run this in your Colab
# (wherever `stf` gets imported) and read the output:
#
#     import spacetimeformer as stf
#     print(stf.__file__)
#     help(stf.model.Spacetimeformer.__init__)
#     help(stf.model.Spacetimeformer.forward)
#
# Confirm d_x != d_y is accepted and that forward() takes a single tensor
# shaped (batch, seq_len, d_x) and returns (batch, out_len, d_y). If either
# of those isn't true, leave STF_USE_EXOG = False -- the try/except below
# will also auto-fall-back to the autoregressive path if construction or a
# forward pass fails, but that's a safety net, not a substitute for reading
# the actual signature first.
STF_USE_EXOG = False   # flip to True only after confirming the API above

try:
    import spacetimeformer as stf
    STF_AVAILABLE = True
except ImportError:
    STF_AVAILABLE = False
    print("⚠️  Spacetimeformer not installed — skipping")

stf_model = None
stf_test_mae = None
stf_pair_names = None
stf_exog_cols = None   # None => pure autoregressive; list => exogenous channels used

if STF_AVAILABLE:
    # Build the pivot from train+val ONLY, exactly matching nf_train's
    # scope. The true test window (last `horizon` days) is never touched
    # here -- it's reserved for Step 8's one-time eval.
    df_trainval = df[df["split"].isin(["train", "val"])]
    stf_df = df_trainval.pivot_table(
        index="Date", columns="group_id", values="target"
    ).dropna(how="any")
    stf_pair_names = list(stf_df.columns)
    n_pairs = len(stf_pair_names)

    # ── Exogenous channel assembly (only attempted if STF_USE_EXOG) ────────
    # gdelt_cols/macro_cols/calendar_cols are pulled from Cell B. They only
    # concat cleanly onto this pivot if they're constant across group_id for
    # a given date (i.e. genuinely date-level, not per-pair) -- gdelt_cols
    # was already filtered to "group-level aggregates only" in Cell B, but
    # macro_cols (rho_/sigma_ DCC-GARCH) is NOT guaranteed to be pair-
    # invariant (DCC-GARCH correlations are often pairwise by construction).
    # Check this explicitly instead of assuming it -- silently averaging or
    # taking-first over a genuinely per-pair signal would be its own leak/
    # distortion, quietly wrong in a way that wouldn't show up as a crash.
    # V5: candidate_exog_cols now uses dcc_garch_leg_cols/macro_leg_cols
    # (each group's own 3 legs) instead of the old blanket macro_cols. Note
    # this makes the per-date-nunique check below trigger EVEN MORE
    # reliably now, by design: leg1_macro etc. are deliberately different
    # per group (that's the whole point of the V5 per-group leg fix), so
    # they will always vary by group_id within a date -- meaning STF_USE_EXOG
    # will correctly and automatically fall back to autoregressive-only for
    # these columns unless this notebook is restructured to a real 3D
    # (date x pair x feature) tensor. calendar_cols/gdelt_cols remain
    # genuinely date-level (constant across groups) and could still work
    # with the simple concat path on their own.
    candidate_exog_cols = calendar_cols + gdelt_cols + dcc_garch_leg_cols + macro_leg_cols
    exog_ok = True
    if STF_USE_EXOG and candidate_exog_cols:
        per_date_nunique = (
            df_trainval.groupby("Date")[candidate_exog_cols]
            .nunique()
            .max()
        )
        non_constant = per_date_nunique[per_date_nunique > 1].index.tolist()
        if non_constant:
            print(f"⚠️  These columns vary by group_id within a single date, so they "
                  f"can't be safely concatenated onto the (date x pair) target pivot "
                  f"without a 3D (date x pair x feature) restructure this notebook "
                  f"doesn't do: {non_constant[:10]}{'...' if len(non_constant) > 10 else ''}\n"
                  f"    Falling back to pure-autoregressive Spacetimeformer (no exog).")
            exog_ok = False

    use_exog = STF_USE_EXOG and exog_ok and bool(candidate_exog_cols)

    if use_exog:
        exog_df = (
            df_trainval.drop_duplicates("Date")
            .set_index("Date")[candidate_exog_cols]
            .reindex(stf_df.index)
        )
        n_exog = exog_df.shape[1]
        print(f"Spacetimeformer (EXOG mode): {n_exog} exogenous channels "
              f"({len(calendar_cols)} calendar + {len(gdelt_cols)} gdelt + "
              f"{len(dcc_garch_leg_cols)} dcc_garch + {len(macro_leg_cols)} macro) "
              f"appended to {n_pairs} target pairs.")
    else:
        n_exog = 0
        exog_df = None
        if STF_USE_EXOG and not candidate_exog_cols:
            print("⚠️  STF_USE_EXOG=True but no calendar/gdelt/macro columns were found "
                  "-- nothing to add, using pure-autoregressive Spacetimeformer.")

    print(f"Spacetimeformer training on {len(stf_df)} dates (train+val only), "
          f"{n_pairs} pairs ({stf_df.index.min().date()} -> {stf_df.index.max().date()})"
          + (f", {n_exog} exog channels" if use_exog else " [pure autoregressive]"))

    d_x = n_pairs + n_exog
    stf_arch_hparams = dict(
        d_x=d_x, d_y=n_pairs, max_seq_len=max_encoder_length, out_len=horizon,
        d_model=64, n_heads=4, e_layers=2, d_layers=1, dropout=0.1,
        embed="spatio-temporal", activation="gelu",
    )

    def _build_and_check():
        m = stf.model.Spacetimeformer(**stf_arch_hparams)
        probe = torch.zeros(1, max_encoder_length, d_x)
        with torch.no_grad():
            probe_out = m(probe)
        if probe_out.shape[-1] != n_pairs:
            raise RuntimeError(
                f"Spacetimeformer output last-dim {probe_out.shape[-1]} != n_pairs "
                f"{n_pairs} with d_x={d_x} != d_y={n_pairs} -- this class does not "
                f"support asymmetric d_x/d_y the way this notebook is assuming."
            )
        return m

    if use_exog:
        try:
            stf_model = _build_and_check()
            stf_exog_cols = candidate_exog_cols
        except Exception as e:
            print(f"⚠️  Exogenous Spacetimeformer construction/forward failed: {e}\n"
                  f"    Falling back to pure-autoregressive Spacetimeformer (d_x=d_y=n_pairs).")
            use_exog = False

    if not use_exog:
        d_x = n_pairs
        stf_arch_hparams = dict(
            d_x=n_pairs, d_y=n_pairs, max_seq_len=max_encoder_length, out_len=horizon,
            d_model=64, n_heads=4, e_layers=2, d_layers=1, dropout=0.1,
            embed="spatio-temporal", activation="gelu",
        )
        stf_model = stf.model.Spacetimeformer(**stf_arch_hparams)
        stf_exog_cols = None

    # ── Assemble X/y tensors ────────────────────────────────────────────────
    target_arr = stf_df.values                       # (dates, n_pairs)
    if use_exog:
        exog_arr = exog_df.values                    # (dates, n_exog)
        full_arr = np.concatenate([target_arr, exog_arr], axis=1)  # (dates, d_x)
    else:
        full_arr = target_arr

    X_full = full_arr[:-horizon]
    y_full = target_arr[horizon:]     # y is ALWAYS target-only (d_y = n_pairs)

    # train+val was already carved to exclude the true test window
    # entirely, so val_window is the only split needed here.
    split_point = X_full.shape[0] - val_window

    X_train = torch.tensor(X_full[:split_point], dtype=torch.float32).unsqueeze(0)
    y_train = torch.tensor(y_full[:split_point], dtype=torch.float32).unsqueeze(0)
    X_val   = torch.tensor(X_full[split_point:], dtype=torch.float32).unsqueeze(0)
    y_val   = torch.tensor(y_full[split_point:], dtype=torch.float32).unsqueeze(0)

    stf_train_hparams = {"lr": 1e-3, "n_epochs": 20, "val_window": val_window, "seed": SEED}
    optimizer = torch.optim.Adam(stf_model.parameters(), lr=stf_train_hparams["lr"])
    n_epochs = stf_train_hparams["n_epochs"]
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    mae_fn, mse_fn = nn.L1Loss(), nn.MSELoss()
    stf_history = {"epoch": [], "train_loss": [], "val_loss": [], "train_mae": [], "val_mae": [], "lr": []}

    # Track and checkpoint the best val_loss epoch instead of just keeping
    # whatever weights exist after the fixed epoch count.
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(n_epochs):
        stf_model.train()
        optimizer.zero_grad()
        out = stf_model(X_train)
        y_train_aligned = y_train[:, :out.shape[1], :]
        train_loss = mse_fn(out, y_train_aligned)
        train_mae = mae_fn(out, y_train_aligned)
        train_loss.backward()
        optimizer.step()

        stf_model.eval()
        with torch.no_grad():
            val_out = stf_model(X_val)
            y_val_aligned = y_val[:, :val_out.shape[1], :]
            val_loss = mse_fn(val_out, y_val_aligned)
            val_mae = mae_fn(val_out, y_val_aligned)

        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_state = {k: v.detach().clone() for k, v in stf_model.state_dict().items()}

        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step()
        for k, v in [("epoch", epoch), ("train_loss", train_loss.item()), ("val_loss", val_loss.item()),
                     ("train_mae", train_mae.item()), ("val_mae", val_mae.item()), ("lr", current_lr)]:
            stf_history[k].append(v)

        if epoch % 5 == 0:
            print(f"  Epoch {epoch:3d} | LR {current_lr:.6f} | "
                  f"train_loss {train_loss.item():.6f} | val_loss {val_loss.item():.6f}")

    # load best-val-loss weights before saving / evaluating
    if best_state is not None:
        stf_model.load_state_dict(best_state)
        print(f"✅ Loaded best checkpoint (val_loss={best_val_loss:.6f})")

    os.makedirs(f"{drive_base}spacetimeformer/", exist_ok=True)
    torch.save(stf_model.state_dict(), f"{drive_base}spacetimeformer/stf_model.pt")
    with open(f"{drive_base}spacetimeformer/history.pkl", "wb") as f:
        pickle.dump(stf_history, f)
    mode_str = f"EXOG ({len(stf_exog_cols)} channels)" if stf_exog_cols else "pure-autoregressive"
    print(f"✅ Spacetimeformer trained on train+val only [{mode_str}], checkpointed on best val_loss")


### Step 8 — Spacetimeformer: true held-out test evaluation

The piece that was completely missing before: this forecasts the real held-out `horizon` days
(never seen during training) and scores against real targets from `df_raw`, mirroring Step 4's
TFT evaluation.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell J2 — Spacetimeformer TRUE test evaluation   (V4)
# ──────────────────────────────────────────────────────────────────────────
if STF_AVAILABLE and stf_model is not None:
    # Build the encoder input from the LAST max_encoder_length days of
    # train+val (i.e. immediately preceding the test window), same
    # boundary TFT and the NF models use.
    stf_full = df.pivot_table(index="Date", columns="group_id", values="target")
    stf_full = stf_full[stf_pair_names].dropna(how="any")

    test_dates = sorted(df[df["split"] == "test"]["Date"].unique())[:horizon]
    if len(test_dates) < horizon:
        print(f"⚠️  Only {len(test_dates)} test dates available (expected {horizon}); "
              f"Spacetimeformer test eval will use what's available.")

    history = stf_full[stf_full.index < test_dates[0]].tail(max_encoder_length)
    if len(history) < max_encoder_length:
        print(f"⚠️  Only {len(history)} encoder days available before test start "
              f"(wanted {max_encoder_length}) -- results may be less reliable.")

    # V4: if training used exogenous channels (stf_exog_cols is not None),
    # the test-window encoder input must be built the same way -- same
    # columns, same order, same date alignment -- or the model sees a
    # different d_x than it was trained on and either crashes or silently
    # misreads which channel is which.
    if stf_exog_cols:
        exog_history = (
            df.drop_duplicates("Date")
            .set_index("Date")[stf_exog_cols]
            .reindex(history.index)
        )
        if exog_history.isna().any().any():
            print("⚠️  Missing exogenous values in the test-window encoder history -- "
                  "check date coverage of calendar/gdelt/macro columns near the test "
                  "boundary before trusting this eval.")
        X_test_arr = np.concatenate([history.values, exog_history.values], axis=1)
    else:
        X_test_arr = history.values

    X_test = torch.tensor(X_test_arr, dtype=torch.float32).unsqueeze(0)

    stf_model.eval()
    with torch.no_grad():
        stf_test_out = stf_model(X_test).squeeze(0).numpy()  # (horizon, n_pairs)

    stf_test_preds = pd.DataFrame(
        stf_test_out[: len(test_dates)], index=test_dates, columns=stf_pair_names
    ).reset_index().melt(id_vars="index", var_name="group_id", value_name="stf_pred")
    stf_test_preds = stf_test_preds.rename(columns={"index": "Date"})

    stf_test_scored = stf_test_preds.merge(
        df_raw[["group_id", "Date", "target"]], on=["group_id", "Date"], how="inner"
    )
    n_expected = len(stf_pair_names) * len(test_dates)
    if len(stf_test_scored) != n_expected:
        print(f"⚠️  Spacetimeformer: matched {len(stf_test_scored)} rows against real "
              f"test targets, expected {n_expected} ({len(stf_pair_names)} pairs x "
              f"{len(test_dates)} test days). Check date alignment.")
    else:
        print(f"✅ Spacetimeformer: all {len(stf_test_scored)} test-window forecasts "
              f"matched to real targets")

    stf_test_scored["abs_err"] = (stf_test_scored["stf_pred"] - stf_test_scored["target"]).abs()
    stf_test_mae = stf_test_scored["abs_err"].mean()
    mode_str = f"EXOG ({len(stf_exog_cols)} channels)" if stf_exog_cols else "pure-autoregressive"
    print(f"   Spacetimeformer [{mode_str}] true held-out test MAE: {stf_test_mae:.5f} "
          f"(n={len(stf_test_scored)})")

    # SAVE predictions + config
    stf_test_scored.to_csv(f"{drive_base}spacetimeformer/test_predictions.csv", index=False)
    with open(f"{drive_base}spacetimeformer/config.json", "w") as f:
        json.dump({
            "n_pairs": len(stf_pair_names), "pair_names": stf_pair_names,
            "max_encoder_length": max_encoder_length, "horizon": horizon,
            "exog_mode": bool(stf_exog_cols),
            "exog_cols": stf_exog_cols,
            # FIX: previously missing -- without these, the saved .pt weights
            # can't be loaded back (Spacetimeformer(**arch) must be rebuilt
            # identically before load_state_dict works).
            "architecture": stf_arch_hparams,
            "training": stf_train_hparams,
        }, f, indent=2)
else:
    stf_test_mae = None


### Step 9 — Evaluate NBEATSx / NHITS / PatchTST on the true test window

`nbeats_preds` / `nhits_preds` / `patchtst_preds` from Step 6 were produced by `.predict()`
immediately after fitting on train+val, so they already forecast exactly the `horizon` days that
follow — i.e. the real test window. Here we match those forecasts (using the q50 / median column)
against the real targets in `df_raw`, the same way Step 8 does for Spacetimeformer.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell L — Shared held-out scoring helper
# ──────────────────────────────────────────────────────────────────────────
def evaluate_on_test(preds_df, model_name, df_raw, quantiles=(0.1, 0.5, 0.9)):
    cols = pick_quantile_cols(preds_df, model_name, quantiles=quantiles)
    q50_col = cols[0.5]

    keep_cols = ["unique_id", "ds", q50_col]
    # Include interval columns when available.
    for q in (0.1, 0.9):
        if q in cols and cols[q] not in keep_cols:
            keep_cols.append(cols[q])

    preds = preds_df[keep_cols].copy().rename(
        columns={"unique_id": "group_id", "ds": "Date", q50_col: f"{model_name}_pred"}
    )
    preds["Date"] = pd.to_datetime(preds["Date"])

    scored = preds.merge(
        df_raw[["group_id", "Date", "target", "split"]],
        on=["group_id", "Date"], how="inner"
    )
    scored = scored[scored["split"] == "test"].copy()

    n_expected = df_raw[df_raw["split"] == "test"].shape[0]
    assert len(scored) == n_expected, (
        f"{model_name}: matched {len(scored)} true test rows, expected {n_expected}. "
        "Check forecast timestamps / future dataframe."
    )

    err = scored[f"{model_name}_pred"] - scored["target"]
    scored["abs_err"] = err.abs()
    scored["sq_err"] = err.pow(2)
    scored["direction_hit"] = (
        np.sign(scored[f"{model_name}_pred"]) == np.sign(scored["target"])
    ).astype(int)

    metrics = {
        "mae": float(scored["abs_err"].mean()),
        "rmse": float(np.sqrt(scored["sq_err"].mean())),
        "direction_accuracy": float(scored["direction_hit"].mean()),
    }

    # Optional 80% interval coverage.
    if 0.1 in cols and 0.9 in cols:
        q10_name, q90_name = cols[0.1], cols[0.9]
        if q10_name in scored.columns and q90_name in scored.columns:
            metrics["q10_q90_coverage"] = float(
                ((scored["target"] >= scored[q10_name]) &
                 (scored["target"] <= scored[q90_name])).mean()
            )

    print(
        f"✅ {model_name}: MAE={metrics['mae']:.6f} | "
        f"RMSE={metrics['rmse']:.6f} | "
        f"direction={metrics['direction_accuracy']:.2%}"
    )

    scored.to_csv(f"{drive_base}{model_name.lower()}/test_scored.csv", index=False)
    return scored, metrics


In [ ]:
# Cell M — NBEATSx true held-out test evaluation
nbeats_scored, nbeats_metrics = evaluate_on_test(nbeats_preds, "NBEATSx", df_raw)
nbeats_mae = nbeats_metrics["mae"]


In [ ]:
# Cell N — NHITS true held-out test evaluation
nhits_scored, nhits_metrics = evaluate_on_test(nhits_preds, "NHITS", df_raw)
nhits_mae = nhits_metrics["mae"]


In [ ]:
# Cell O — PatchTST true held-out test evaluation
patchtst_scored, patchtst_metrics = evaluate_on_test(patchtst_preds, "PatchTST", df_raw)
patchtst_mae = patchtst_metrics["mae"]


### Step 10 — Ensemble the q50 forecasts (NBEATSx + NHITS + PatchTST)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell K — Robust ensemble: mean + median, no test-tuned weights
# ──────────────────────────────────────────────────────────────────────────
def q50_frame(preds_df, model_name):
    cols = pick_quantile_cols(preds_df, model_name, quantiles=(0.5,))
    q50_col = cols[0.5]
    return preds_df[["unique_id", "ds", q50_col]].rename(
        columns={q50_col: f"{model_name}_q50"}
    )

merged = (
    q50_frame(nbeats_preds, "NBEATSx")
    .merge(q50_frame(nhits_preds, "NHITS"), on=["unique_id", "ds"], how="inner")
    .merge(q50_frame(patchtst_preds, "PatchTST"), on=["unique_id", "ds"], how="inner")
)

model_cols = ["NBEATSx_q50", "NHITS_q50", "PatchTST_q50"]
merged["ensemble_mean_q50"] = merged[model_cols].mean(axis=1)
merged["ensemble_median_q50"] = merged[model_cols].median(axis=1)
merged["model_spread"] = merged[model_cols].max(axis=1) - merged[model_cols].min(axis=1)

_truth = df_raw[df_raw["split"] == "test"][["group_id", "Date", "target"]].copy()
_truth["Date"] = pd.to_datetime(_truth["Date"])
ensemble_scored = merged.rename(columns={"unique_id": "group_id", "ds": "Date"})
ensemble_scored["Date"] = pd.to_datetime(ensemble_scored["Date"])
ensemble_scored = ensemble_scored.merge(_truth, on=["group_id", "Date"], how="inner")

assert len(ensemble_scored) == len(_truth),     "Ensemble timestamps do not fully match the held-out test set."

ensemble_metrics = {}
for label, pred_col in [
    ("EnsembleMean", "ensemble_mean_q50"),
    ("EnsembleMedian", "ensemble_median_q50"),
]:
    e = ensemble_scored[pred_col] - ensemble_scored["target"]
    ensemble_metrics[label] = {
        "mae": float(e.abs().mean()),
        "rmse": float(np.sqrt((e ** 2).mean())),
        "direction_accuracy": float(
            (np.sign(ensemble_scored[pred_col]) == np.sign(ensemble_scored["target"])).mean()
        ),
    }
    print(
        f"{label}: MAE={ensemble_metrics[label]['mae']:.6f} | "
        f"RMSE={ensemble_metrics[label]['rmse']:.6f} | "
        f"direction={ensemble_metrics[label]['direction_accuracy']:.2%}"
    )

merged.to_csv(f"{drive_base}ensemble/predictions.csv", index=False)
ensemble_scored.to_csv(f"{drive_base}ensemble/test_scored.csv", index=False)


### Step 11 — Unified leaderboard

TFT, NBEATSx, NHITS, PatchTST, and Spacetimeformer test performance side by side — all computed
from the *same* `horizon` held-out days.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Cell P — Unified held-out leaderboard + zero-change baseline
# ──────────────────────────────────────────────────────────────────────────
def _extract_metric(test_metrics, needle):
    if not test_metrics:
        return None
    m = test_metrics[0]
    for key, value in m.items():
        if needle.lower() in key.lower():
            try:
                return float(value)
            except Exception:
                return value
    return None

test_truth = df_raw[df_raw["split"] == "test"]["target"]
zero_mae = float(test_truth.abs().mean())
zero_rmse = float(np.sqrt((test_truth ** 2).mean()))

tft_mae = _extract_metric(test_metrics, "mae")
tft_rmse = _extract_metric(test_metrics, "rmse")

_stf_features = (
    f"price+exog ({len(stf_exog_cols)} exog ch.)" if stf_exog_cols
    else "target history only"
)

leaderboard_rows = [
    {
        "model": "ZeroChangeBaseline",
        "test_mae": zero_mae,
        "test_rmse": zero_rmse,
        "direction_accuracy": np.nan,
        "features": "predict 0 for every future z-score change",
    },
    {
        "model": "TFT",
        "test_mae": tft_mae,
        "test_rmse": tft_rmse,
        "direction_accuracy": np.nan,
        "features": f"{len(hist_exog)} hist + {len(calendar_cols)} future-calendar + group embedding",
    },
    {
        "model": "NBEATSx",
        "test_mae": nbeats_metrics["mae"],
        "test_rmse": nbeats_metrics["rmse"],
        "direction_accuracy": nbeats_metrics["direction_accuracy"],
        "features": str(nf_feature_mode["NBEATSx"]),
    },
    {
        "model": "NHITS",
        "test_mae": nhits_metrics["mae"],
        "test_rmse": nhits_metrics["rmse"],
        "direction_accuracy": nhits_metrics["direction_accuracy"],
        "features": str(nf_feature_mode["NHITS"]),
    },
    {
        "model": "PatchTST",
        "test_mae": patchtst_metrics["mae"],
        "test_rmse": patchtst_metrics["rmse"],
        "direction_accuracy": patchtst_metrics["direction_accuracy"],
        "features": str(nf_feature_mode["PatchTST"]),
    },
    {
        "model": "EnsembleMean",
        "test_mae": ensemble_metrics["EnsembleMean"]["mae"],
        "test_rmse": ensemble_metrics["EnsembleMean"]["rmse"],
        "direction_accuracy": ensemble_metrics["EnsembleMean"]["direction_accuracy"],
        "features": "equal mean of NBEATSx/NHITS/PatchTST q50",
    },
    {
        "model": "EnsembleMedian",
        "test_mae": ensemble_metrics["EnsembleMedian"]["mae"],
        "test_rmse": ensemble_metrics["EnsembleMedian"]["rmse"],
        "direction_accuracy": ensemble_metrics["EnsembleMedian"]["direction_accuracy"],
        "features": "median of NBEATSx/NHITS/PatchTST q50",
    },
]
if stf_test_mae is not None:
    leaderboard_rows.append({
        "model": "Spacetimeformer",
        "test_mae": float(stf_test_mae),
        "test_rmse": np.nan,
        "direction_accuracy": np.nan,
        "features": _stf_features,
    })

leaderboard = pd.DataFrame(leaderboard_rows).dropna(subset=["test_mae"])
leaderboard["beats_zero_mae"] = leaderboard["test_mae"] < zero_mae
leaderboard = leaderboard.sort_values("test_mae").reset_index(drop=True)

print("\n════════════════════════════════════════════════════════════════════")
print(" UNIFIED LEADERBOARD — TRUE HELD-OUT FINAL 5 BUSINESS DAYS / GROUP")
print("════════════════════════════════════════════════════════════════════")
print(leaderboard.to_string(index=False))

os.makedirs(f"{drive_base}leaderboard/", exist_ok=True)
leaderboard.to_csv(f"{drive_base}leaderboard/leaderboard.csv", index=False)
with open(f"{drive_base}leaderboard.pkl", "wb") as f:
    pickle.dump(leaderboard, f)

best_row = leaderboard.iloc[0]
print(
    f"\nBest held-out MAE: {best_row['model']} = {best_row['test_mae']:.6f}. "
    f"Zero baseline = {zero_mae:.6f}."
)


### Step 12 — Manifest + consolidated hyperparameters

Writes `manifest.json` (pointing at the actual scored test files, not just raw predictions) and
`hyperparameters.json` (every model's architecture/training config in one place, alongside the
seed used) so this run's leaderboard is fully auditable later.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# Cell Q — V7 manifest + consolidated configuration
# ══════════════════════════════════════════════════════════════════════════
manifest = {
    "run": {
        "pipeline_version": "V7_actual_82col",
        "seed": SEED,
        "csv_source": CSV_PATH,
        "target_definition": "target_t = zscore_t - zscore_(t-1)",
        "horizon": horizon,
        "max_encoder_length": max_encoder_length,
        "val_window": val_window,
        "n_groups": int(df["group_id"].nunique()),
        "historical_features": hist_exog,
        "known_future_features": calendar_cols,
        "explicitly_excluded_leak_columns": LEAK_COLS,
        "neuralforecast_feature_support": nf_feature_mode,
    },
    "data": {
        "full_dataset": f"{DATA_DIR}full_dataset.parquet",
        "train": f"{DATA_DIR}train.parquet",
        "val": f"{DATA_DIR}val.parquet",
        "test": f"{DATA_DIR}test.parquet",
        "split_config": f"{DATA_DIR}split_config.json",
    },
    "models": {
        "tft": {
            "checkpoint": best_tft_path,
            "dataset_params": f"{drive_base}tft/dataset_params.pkl",
            "hyperparameters": f"{drive_base}tft/hyperparameters.json",
            "test_predictions": f"{drive_base}tft/test_predictions.pkl",
            "test_metrics": f"{drive_base}tft/test_metrics.pkl",
            "test_mae": tft_mae,
        },
        "nbeats": {
            "model_dir": f"{drive_base}nbeats/",
            "test_scored": f"{drive_base}nbeats/test_scored.csv",
            "metrics": nbeats_metrics,
        },
        "nhits": {
            "model_dir": f"{drive_base}nhits/",
            "test_scored": f"{drive_base}nhits/test_scored.csv",
            "metrics": nhits_metrics,
        },
        "patchtst": {
            "model_dir": f"{drive_base}patchtst/",
            "test_scored": f"{drive_base}patchtst/test_scored.csv",
            "metrics": patchtst_metrics,
        },
        "ensemble": {
            "predictions": f"{drive_base}ensemble/predictions.csv",
            "test_scored": f"{drive_base}ensemble/test_scored.csv",
            "metrics": ensemble_metrics,
        },
        "spacetimeformer": {
            "available": bool(STF_AVAILABLE),
            "test_mae": None if stf_test_mae is None else float(stf_test_mae),
            "exog_cols": stf_exog_cols,
        },
    },
    "leaderboard": {
        "csv": f"{drive_base}leaderboard/leaderboard.csv",
        "zero_change_test_mae": zero_mae,
    },
}

with open(f"{drive_base}manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

with open(f"{drive_base}hyperparameters.json", "w") as f:
    json.dump({
        "tft": tft_hparams,
        "neuralforecast": nf_hparams,
    }, f, indent=2, default=str)

print(f"✅ V7 manifest: {drive_base}manifest.json")
print(f"✅ Consolidated hyperparameters: {drive_base}hyperparameters.json")
